In [1]:
import json
import re
from google import genai

In [ ]:
## 입력 PATHS
PDF_PATH = r".\data\영수증_PDF\3. 2025-11_지출결의영수증_이래건.pdf"

## 출력 PATHS
JSON_PATH = r".\output\3. 2025-11_지출결의영수증_이래건.json"

## LLM MODEL 설정
MODEL_NAME =  "gemini-3-pro-preview"

In [3]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [5]:
import shutil
import os

# 1. 클라이언트 초기화
# 환경 변수에서 GEMINI_API_KEY를 자동으로 가져옵니다.
client = genai.Client()

# --- 2. File API를 사용하여 PDF 파일 업로드 ---
print(f"'{PDF_PATH}' 파일을 업로드하는 중...")

# 한글 파일명으로 인한 UnicodeEncodeError 방지를 위해 임시 파일(ASCII 이름)로 복사 후 업로드
temp_file_path = "temp_upload.pdf"
shutil.copy(PDF_PATH, temp_file_path)

try:
    # 파일을 업로드하고 File 객체를 반환받습니다.
    uploaded_file = client.files.upload(file=temp_file_path)
    print(f"업로드 완료. 파일 이름: {uploaded_file.name}")
    print(f"MIME 타입: {uploaded_file.mime_type}")
finally:
    # 임시 파일 삭제
    if os.path.exists(temp_file_path):
        os.remove(temp_file_path)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


'.\data\영수증_PDF\1. 2025-11_지출결의영수증_지수현.pdf' 파일을 업로드하는 중...
업로드 완료. 파일 이름: files/8t6dspw62iiv
MIME 타입: application/pdf


In [6]:
# --- 3. Gemini 모델에 요청 보내기 ---
prompt = """
# 영수증 필수 추출 항목 (Extract Items)

`Verify-Rule.md`에 정의된 검증 규칙을 수행하기 위해 OCR(광학 문자 인식) 또는 파싱을 통해 영수증 이미지/파일에서 반드시 추출해야 하는 데이터 항목들입니다.
다음 항목을 추출하여 JSON 형식으로 반환해 주세요.

## 1. 기본 결제 정보 (Basic Transaction Info)
가장 핵심적인 검증 대상이며, 지출 내역 매칭에 사용됩니다.

- **거래 일자 (Date)**
  - *형식:* YYYY-MM-DD
  - *용도:* 신청 내역 날짜 비교, 주말/공휴일 사용 여부 확인
- **거래 시간 (Time)**
  - *형식:* HH:MM:SS (또는 HH:MM)
  - *용도:* 심야 시간 사용 여부, 분할 결제(시간차 공격) 탐지
- **합계 금액 (Total Amount)**
  - *형식:* 숫자 (원화)
  - *용도:* 신청 금액 일치 여부 확인
- **승인 번호 (Approval Code / Authorization No)**
  - *형식:* 문자열/숫자
  - *용도:* 중복 제출 방지, 유니크 키 식별, 법인카드 승인 내역 대조

## 2. 가맹점 정보 (Vendor Info)
지출처의 적격성과 실제 방문 여부를 판단합니다.

- **가맹점 명 (Vendor Name / Merchant Name)**
  - *용도:* 신청 내역의 사용처와 비교
- **사업자 등록 번호 (Tax ID / Business Registration No)**
  - *형식:* 000-00-00000
  - *용도:* 세무 증빙 유효성 확인, 휴폐업 조회
- **가맹점 주소 (Vendor Address)**
  - *용도:* 실제 가맹점 위치 확인 (선택적)
- **전화 번호 (Phone Number)**
  - *용도:* 가맹점 식별 보조 정보 (선택적)

## 3. 상세 결제 내역 (Payment Details)
금액의 구성 요소와 결제 수단을 검증합니다.

- **공급가액 (Supply Value / Net Amount)**
  - *용도:* 금액 무결성 검증 (공급가액 + 부가세 = 합계)
- **부가세 (VAT / Tax)**
  - *용도:* 금액 무결성 검증, 매입세액 공제 확인
- **봉사료 (Service Charge)**
  - *용도:* 금액 합산 검증 (있을 경우)
- **카드 번호 (Card Number)**
  - *형식:* 마스킹 된 번호 (예: 1234-****-****-5678)
  - *용도:* 법인카드 사용 여부 확인

## 4. 품목 상세 (Line Items)
구매한 물품이 회사 규정에 부합하는지 확인합니다.

- **품목 명 (Item Name)**
  - *용도:* 금지 품목(주류, 담배 등) 키워드 검색
- **단가 및 수량 (Unit Price & Quantity)**
  - *용도:* 상세 금액 계산 검증
- **품목별 금액 (Item Total)**
  - *용도:* 품목 합계가 총 합계와 일치하는지 확인

## 5. 메타 데이터 (Meta Data)
- **OCR 신뢰도 (Confidence Score)**
  - *용도:* 데이터의 정확성 판단, 수동 검토 필요 여부 플래그
    """

In [7]:
# 모델에 업로드된 파일과 텍스트 프롬프트를 함께 전달합니다.
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[
        uploaded_file,
        prompt
    ]
)

# --- 4. 응답을 json 파일로 저장 ---
# Gemini API의 응답 텍스트를 가져옴
raw = response.text

# 정규식을 사용하여 마크다운 코드 블록(```json ... ``` 또는 ``` ... ```)을 제거
# re.DOTALL: 개행 문자를 포함한 모든 문자를 매칭
# re.IGNORECASE: 대소문자 구분 없이 매칭
m = re.match(r"^```(?:json)?\s*(.*)\s*```$", raw, flags=re.DOTALL | re.IGNORECASE)

# 매칭된 그룹(코드 블록 내부 내용)이 있으면 추출, 없으면 원본 텍스트 사용
clean = m.group(1) if m else raw


# Gemini 응답이 JSON 문자열이라면 바로 저장
parsed_json = json.loads(clean)
with open(JSON_PATH, "w", encoding="utf-8") as json_file:
    json.dump(parsed_json, json_file, ensure_ascii=False, indent=2)
print(f"\n결과가 '{JSON_PATH}' 파일로 저장되었습니다.")


결과가 '.\output\1. 2025-11_지출결의영수증_지수현.json' 파일로 저장되었습니다.


In [ ]:
print(len(parsed_json))